In [112]:
# Load the Kedro IPython extension
%load_ext kedro.ipython

The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython


In [113]:
import numpy as np
import plotly.graph_objects as go

In [114]:
var = "tct"
second_stage_shap_value = catalog.load("second_stage_shap_value_" + var)
second_stage_customer_id = catalog.load("second_stage_customer_id_" + var)
second_stage_predictions = catalog.load("second_stage_predictions_" + var)
second_stage_feature_names = catalog.load("second_stage_feature_names_" + var)
second_stage_X = catalog.load("second_stage_X_" + var)
second_stage_explainer = catalog.load("second_stage_explainer_" + var)

[07/03/25 18:51:14] INFO     Loading data from second_stage_shap_value_tct (PickleDataset)...   ]8;id=821455;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=128121;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py#403\403]8;;\

                    INFO     Loading data from second_stage_customer_id_tct (PickleDataset)...  ]8;id=620235;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=373082;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py#403\403]8;;\

                    INFO     Loading data from second_stage_predictions_tct                     ]8;id=393939;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=38444;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py#403\403]8;;\
                             (PolarsParquetDataset)...                                                             

                    INFO     Loading data from second_stage_feature_names_tct                   ]8;id=479181;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=642376;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py#403\403]8;;\
                             (PickleDataset)...                                                                    

                    INFO     Loading data from second_stage_X_tct (PickleDataset)...            ]8;id=963715;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=252343;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py#403\403]8;;\

                    INFO     Loading data from second_stage_explainer_tct (PickleDataset)...    ]8;id=126039;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=592217;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py#403\403]8;;\

In [115]:
feature_dict = {
    "scaler__fleet_size": "Fleet Size",
    "scaler__c_yearly_volumen": "TCT Annual Volume",
    "scaler__c_yearly_network_volumen": "Network Annual Volume",
    "scaler__c_yearly_avg_volume_per_trx": "TCT Avg Volume per Transaction (Annual)",
    "scaler__region_diversity_index": "Regional Diversity Index",
    "scaler__c_weighted_competitive_index": "Weighted Competitiveness Index",
    "scaler__c_tct_yearly_perc_volume": "TCT Share of Annual Volume (%)",
    "scaler__share_customer_region_13.0": "Share of Region 13",
    "scaler__share_customer_region_7.0": "Share of Region 7",
    "scaler__share_customer_region_3.0": "Share of Region 3",
    "scaler__share_customer_region_6.0": "Share of Region 6",
    "scaler__share_customer_region_16.0": "Share of Region 16",
    "scaler__share_customer_region_10.0": "Share of Region 10",
    "scaler__share_customer_region_15.0": "Share of Region 15",
    "scaler__share_customer_region_14.0": "Share of Region 14",
    "scaler__share_customer_region_8.0": "Share of Region 8",
    "scaler__share_customer_region_4.0": "Share of Region 4",
    "scaler__share_customer_region_9.0": "Share of Region 9",
    "scaler__share_customer_region_2.0": "Share of Region 2",
    "share_customer_region_5.0": "Share of Region 5",
    "scaler__share_customer_region_1.0": "Share of Region 1",
    "scaler__share_customer_region_11.0": "Share of Region 11",
    "scaler__share_customer_region_12.0": "Share of Region 12",
}

In [116]:
def build_shap_waterfall_plot(
    customer_idx, shap_values, X, feature_names, explainer, max_display=10
):
    shap_exp = shap_values[customer_idx]
    base_value = explainer.expected_value
    shap_vals = shap_exp
    data_vals = X.iloc[customer_idx].values

    # Sort top features by absolute SHAP
    sorted_idx = np.argsort(np.abs(shap_vals))[::-1][:max_display]
    sorted_features = [feature_names[i] for i in sorted_idx]
    sorted_shap = shap_vals[sorted_idx]
    sorted_data = data_vals[sorted_idx]

    labels = [f"{f} = {round(v, 2)}" for f, v in zip(sorted_features, sorted_data)]

    # Cumulative positions from base value
    x_start = [base_value]
    for val in sorted_shap[:-1]:
        x_start.append(x_start[-1] + val)
    x_end = x_start[1:] + [base_value + sum(sorted_shap)]

    base_vals = x_start
    widths = sorted_shap
    colors = ["green" if v > 0 else "red" for v in widths]

    fig = go.Figure()

    for i, (label, w, base, color) in enumerate(zip(labels, widths, base_vals, colors)):
        fig.add_trace(
            go.Bar(
                y=[label],
                x=[w],
                orientation="h",
                base=base,
                marker_color=color,
                name=label,
                hovertemplate=f"{label}<br>SHAP: {w:.3f}<extra></extra>",
            )
        )

    final_prediction = base_value + sum(widths)

    # Add base value line
    fig.add_vline(
        x=base_value,
        line=dict(dash="dot", color="blue"),
        annotation_text=f"Base value: {base_value:.2f}",
        annotation_position="bottom left",
    )

    # Add final prediction line
    fig.add_vline(
        x=final_prediction,
        line=dict(dash="dot", color="black"),
        annotation_text=f"Prediction: {final_prediction:.2f}",
        annotation_position="top left",
    )

    fig.update_layout(
        title="SHAP Waterfall",
        xaxis_title="Model Output",
        yaxis=dict(autorange="reversed"),  # top to bottom
        height=600,
        showlegend=False,
        margin=dict(l=150, t=60),
    )

    return fig

In [117]:
from dash import Dash, Input, Output, dcc, html

customer_ids = second_stage_customer_id.tolist()
customer_id_to_index = {cid: i for i, cid in enumerate(customer_ids)}

app = Dash(__name__)

app.layout = html.Div(
    [
        dcc.Dropdown(
            id="customer_dropdown",
            options=[{"label": str(cid), "value": cid} for cid in customer_ids],
            value=customer_ids[0],  # or any default
            placeholder="Select Customer Index",
        ),
        dcc.Graph(id="shap_waterfall_plot"),
    ]
)


@app.callback(
    Output("shap_waterfall_plot", "figure"), Input("customer_dropdown", "value")
)
def update_waterfall_plot(selected_customer_id):
    customer_idx = customer_id_to_index[selected_customer_id]
    fig = build_shap_waterfall_plot(
        customer_idx,
        second_stage_shap_value,
        second_stage_X,
        second_stage_feature_names,
        second_stage_explainer,
        max_display=30,
    )
    return fig


if __name__ == "__main__":
    app.run(debug=True)

<IPython.lib.display.IFrame object at 0x32f3ebd90>

In [118]:
region_zone_map = {
    "Region Norte": [1, 2, 3, 4, 15],
    "Region Centro": [5, 6, 7, 13],
    "Region Sur": [8, 9, 10, 14, 11, 12, 16],
}


def group_shap_regions(shap_values_row, feature_names, data_values_row):
    zone_shap = {}
    zone_values = {}
    used_indices = set()

    for zone, region_ids in region_zone_map.items():
        region_features = [f"scaler__share_customer_region_{i}" for i in region_ids]
        indices = [
            feature_names.tolist().index(f)
            for f in region_features
            if f in feature_names
        ]
        used_indices.update(indices)
        zone_shap[zone] = sum(shap_values_row[i] for i in indices)
        zone_values[zone] = sum(data_values_row[i] for i in indices)

    return zone_shap, zone_values, used_indices


def build_shap_waterfall_with_regions(
    customer_idx, shap_values, X, feature_names, explainer, max_display=10
):
    shap_exp = shap_values[customer_idx]
    base_value = explainer.expected_value
    shap_vals = shap_exp
    data_vals = X.iloc[customer_idx].values

    # Step 1: Group SHAP values for regions into zones
    zone_shap, zone_values, used_region_indices = group_shap_regions(
        shap_vals, feature_names, data_vals
    )

    # Step 2: Identify remaining (non-region) features
    remaining_indices = [
        i for i in range(len(shap_vals)) if i not in used_region_indices
    ]

    # Get top N features (by absolute SHAP) from the rest
    top_rest_idx = sorted(
        remaining_indices, key=lambda i: abs(shap_vals[i]), reverse=True
    )[: max_display - len(zone_shap)]

    # Combine zones + individual top features
    combined = [
        (f, v, d)
        for f, v, d in zip(
            list(zone_shap.keys())
            + [feature_dict[feature_names[i]] for i in top_rest_idx],
            list(zone_shap.values()) + [shap_vals[i] for i in top_rest_idx],
            [zone_values[f] for f in zone_shap.keys()]
            + [data_vals[i] for i in top_rest_idx],
        )
    ]

    # Sort by absolute SHAP
    combined_sorted = sorted(combined, key=lambda x: abs(x[1]), reverse=True)

    final_shap_values = [v for _, v, _ in combined_sorted]

    labels = [
        f"{f} = {round(d, 2)}" if d is not None else f"{f}"
        for f, _, d in combined_sorted
    ]

    # Build horizontal waterfall
    x_start = [base_value]
    for val in final_shap_values[:-1]:
        x_start.append(x_start[-1] + val)
        x_end = x_start[1:] + [base_value + sum(final_shap_values)]

    base_vals = x_start
    widths = final_shap_values
    colors = ["green" if v > 0 else "red" for v in widths]

    fig = go.Figure()

    for i, (label, w, base, color) in enumerate(zip(labels, widths, base_vals, colors)):
        fig.add_trace(
            go.Bar(
                y=[label],
                x=[w],
                orientation="h",
                base=base,
                marker_color=color,
                name=label,
                hovertemplate=f"{label}<br>SHAP: {w:.3f}<extra></extra>",
            )
        )

    final_prediction = base_value + sum(widths)

    # Add lines for base and prediction
    fig.add_vline(
        x=base_value,
        line=dict(dash="dot", color="blue"),
        annotation_text=f"Base value: {base_value:.2f}",
        annotation_position="bottom left",
    )

    fig.add_vline(
        x=final_prediction,
        line=dict(dash="dot", color="black"),
        annotation_text=f"Prediction: {final_prediction:.2f}",
        annotation_position="top left",
    )

    fig.update_layout(
        title="SHAP Waterfall (with Region Zones)",
        xaxis_title="Model Output",
        yaxis=dict(autorange="reversed"),
        height=600,
        showlegend=False,
        margin=dict(l=150, t=60),
    )

    return fig


In [119]:
from dash import Dash, Input, Output, dcc, html

customer_ids = second_stage_customer_id.tolist()
customer_id_to_index = {cid: i for i, cid in enumerate(customer_ids)}

app = Dash(__name__)

app.layout = html.Div(
    [
        dcc.Dropdown(
            id="customer_dropdown",
            options=[{"label": str(cid), "value": cid} for cid in customer_ids],
            value=customer_ids[0],  # or any default
            placeholder="Select Customer Index",
        ),
        dcc.Graph(id="shap_waterfall_plot"),
        dcc.Graph(id="shap_waterfall_plot_all"),
    ]
)


@app.callback(
    Output("shap_waterfall_plot", "figure"), Input("customer_dropdown", "value")
)
def update_waterfall_plot_(selected_customer_id):
    customer_idx = customer_id_to_index[selected_customer_id]
    fig = build_shap_waterfall_with_regions(
        customer_idx,
        second_stage_shap_value,
        second_stage_X,
        second_stage_feature_names,
        second_stage_explainer,
        max_display=10,
    )
    return fig


@app.callback(
    Output("shap_waterfall_plot_all", "figure"), Input("customer_dropdown", "value")
)
def update_waterfall_plot_all(selected_customer_id):
    customer_idx = customer_id_to_index[selected_customer_id]
    fig = build_shap_waterfall_plot(
        customer_idx,
        second_stage_shap_value,
        second_stage_X,
        second_stage_feature_names,
        second_stage_explainer,
        max_display=30,
    )
    return fig


if __name__ == "__main__":
    app.run(debug=True, host="0.0.0.0", port=9090)

<IPython.lib.display.IFrame object at 0x32fb1aa50>

In [120]:
second_stage_X

,c_yearly_network_volumen,c_weighted_competitive_index,share_customer_region_13,share_customer_region_7,share_customer_region_3,share_customer_region_6,share_customer_region_16,share_customer_region_10,share_customer_region_15,share_customer_region_14,share_customer_region_8,share_customer_region_4,share_customer_region_9,share_customer_region_2,share_customer_region_5,share_customer_region_1,share_customer_region_11,share_customer_region_12
0,1.037713e+05,7.228234,0.007686,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.992314,0.000000,0.000000,0.000000
1,2.680480e+04,13.732275,0.875201,0.077391,0.000000,0.023391,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.024017,0.000000,0.000000,0.000000
2,5.220727e+04,8.284663,0.000000,0.001410,0.000000,0.000000,0.000936,0.000000,0.000000,0.000000,0.997655,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,7.397216e+04,35.804589,0.738136,0.167607,0.000000,0.005335,0.000000,0.011395,0.000000,0.022489,0.034363,0.000000,0.020676,0.000000,0.000000,0.000000,0.000000,0.000000
4,3.502058e+04,8.545970,0.000000,0.974633,0.000000,0.008690,0.013219,0.000000,0.000000,0.000000,0.003459,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7872,1.326894e+05,7.355890,0.058203,0.014424,0.091131,0.143978,0.001593,0.078607,0.000000,0.000000,0.002190,0.058433,0.001184,0.257568,0.013635,0.015862,0.017324,0.245866
7873,1.358600e+06,10.301905,0.065306,0.003272,0.070904,0.000000,0.000094,0.206740,0.080443,0.000040,0.000000,0.168368,0.125418,0.133321,0.000116,0.125584,0.000000,0.020394
7874,5.209149e+03,14.852277,0.220886,0.056765,0.000000,0.440100,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.282248,0.000000,0.000000,0.000000
7875,4.899200e+03,45.184534,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
